# Entrenamiento de modelos: dengue pediátrico

Este notebook prepara el dataset, documenta la limpieza y entrena los dos modelos solicitados: regresión logística y red neuronal artificial.

In [ ]:
from pathlib import Path
import sys
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data_processing import prepare_training_data

DATA_PATH = ROOT / "MASTER_CHART_F1000Research.xlsx"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)

## 1. Carga y revisión inicial

In [ ]:
raw_df = pd.read_excel(DATA_PATH)
print(f"Dimensiones originales: {raw_df.shape}")
raw_df.head()

## 2. Limpieza y creación de variables

La función compartida aplica la misma transformación que utilizará Flask:

- Normalización de nombres de columnas.
- Eliminación de duplicados.
- Conversión de códigos binarios 1/2 a 1/0.
- Eliminación de columnas con al menos 30% de nulos o con una categoría dominante de al menos 95%.
- Creación de rango de WBC, relación HB/PCV, cantidad de síntomas y cantidad de tratamientos.
- La imputación de valores numéricos se realiza después dentro del pipeline, usando únicamente el conjunto de entrenamiento.

In [ ]:
X, y, columns_to_drop, cleaning_summary = prepare_training_data(raw_df)
print(json.dumps(cleaning_summary, indent=2, default=str))
print("\nVariables finales:")
print(list(X.columns))
print("\nValores faltantes antes de imputar:")
print(X.isna().sum()[X.isna().sum() > 0])

In [ ]:
print("Distribución de la variable objetivo:")
print(y.value_counts().rename(index={0: "Dengue no complicado", 1: "Dengue complicado"}))
X.describe().T

## 3. División estratificada

Se reserva el 20% para prueba y se conserva la proporción de las dos clases.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Entrenamiento: {X_train.shape}; prueba: {X_test.shape}")

## 4. Entrenamiento de los modelos

Ambos modelos reciben imputación por mediana y estandarización. La regresión logística usa pesos balanceados porque la clase de dengue complicado es minoritaria.

In [ ]:
common_steps = [
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]

logistic_model = Pipeline(common_steps + [
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))
])

neural_model = Pipeline(common_steps + [
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(16, 8),
        activation="relu",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1500,
        early_stopping=True,
        validation_fraction=0.20,
        n_iter_no_change=30,
        random_state=RANDOM_STATE,
    ))
])

models = {"logistic": logistic_model, "neural": neural_model}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: entrenado")

## 5. Evaluación en el conjunto de prueba

In [ ]:
def evaluate_model(model, X_eval, y_eval):
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]
    return {
        "accuracy": float(accuracy_score(y_eval, predictions)),
        "precision": float(precision_score(y_eval, predictions, zero_division=0)),
        "recall": float(recall_score(y_eval, predictions, zero_division=0)),
        "f1": float(f1_score(y_eval, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_eval, probabilities)),
        "confusion_matrix": confusion_matrix(y_eval, predictions, labels=[0, 1]).tolist(),
        "classification_report": classification_report(y_eval, predictions, target_names=["No complicado", "Complicado"], zero_division=0),
    }

metrics = {}
for name, model in models.items():
    metrics[name] = evaluate_model(model, X_test, y_test)
    print(f"\n=== {name} ===")
    print(metrics[name]["classification_report"])
    print("Matriz de confusión:")
    print(np.array(metrics[name]["confusion_matrix"]))
    print(f"ROC-AUC: {metrics[name]['roc_auc']:.3f}")

## 6. Guardado de artefactos para Flask

Se guarda un único archivo con los pipelines, el orden de variables y la información de limpieza. Así la aplicación no vuelve a entrenar los modelos y evita inconsistencias entre notebook y página web.

In [ ]:
bundle = {
    "models": models,
    "feature_columns": list(X.columns),
    "columns_to_drop": columns_to_drop,
    "summary": cleaning_summary,
    "test_metrics": metrics,
    "target_mapping": {0: "Dengue no complicado", 1: "Dengue complicado"},
    # La aplicación usa esta referencia para procesar GROUPS de archivos cargados.
    "encode_target": lambda series: series.map({1: 0, 2: 1}).fillna(series).astype(int),
}

# Las funciones lambda no siempre son serializables de forma portable. Se elimina
# del artefacto y Flask usa la misma función compartida del módulo.
bundle.pop("encode_target")
joblib.dump(bundle, MODEL_DIR / "model_bundle.joblib")
with open(MODEL_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2, ensure_ascii=False)
print(f"Artefactos guardados en: {MODEL_DIR}")

In [ ]:
# Verificación rápida con el primer paciente del conjunto de prueba
sample = X_test.iloc[[0]]
for name, model in models.items():
    prediction = int(model.predict(sample)[0])
    probability = float(model.predict_proba(sample)[0, 1])
    print(name, "=>", "Dengue complicado" if prediction else "Dengue no complicado", f"({probability:.2%})")